# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.

Following code is all just copy paste from the exercise on Monday.

In [1]:
import open3d as o3d
import numpy as np
import copy

# Short alias, so the registration calls below stay readable
reg = o3d.pipelines.registration

# RANSAC picks random samples, seeding reduces the run-to-run variation
# (open3d runs RANSAC multithreaded, so some variation remains)
o3d.utility.random.seed(42)

# helper function for drawing
# If you want it to be more clear set recolor=True
def draw_registrations(source, target, transformation = None, recolor = False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

We have been running the same code found below just changing the file name in the reading line. The cars align very easily. The office room aligns a little worse with fit around .5 instead of 0.7 for the cars.

To avoid editing the file name between runs, the cells below now loop over both pairs: `r1.pcd`/`r2.pcd` (office room) and `car1.ply`/`car2.ply` (car).

In [2]:
pairs = [
    ("global_registration/r1.pcd", "global_registration/r2.pcd"),
    ("global_registration/car1.ply", "global_registration/car2.ply"),
]

# Used for downsampling.
voxel_size = 0.05

The above voxel_size is a good middle ground. Going to 0.1 gives unacceptable alignment, and going to 0.01 makes runtime too high (over 2 minutes compared to 10 seconds at 0.05).

In [3]:
####
# Downsample and find features here
####
# Same steps and parameters as before, wrapped in a function so it can be
# applied to both point clouds of every pair.
def preprocess_point_cloud(pcd, voxel_size):
    pcd_d = pcd.voxel_down_sample(voxel_size=voxel_size)

    radius_normal = voxel_size * 2
    radius_feature = voxel_size * 5

    pcd_d.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=radius_normal,
            max_nn=30
        )
    )

    pcd_features = reg.compute_fpfh_feature(
        pcd_d,
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=radius_feature,
            max_nn=100
        )
    )
    return pcd_d, pcd_features

In [4]:
####
# Call RANSAC here
####

point_to_point =  reg.TransformationEstimationPointToPoint(False)

# Parameters
threshold = 0.02

results = {}
for source_file, target_file in pairs:
    source = o3d.io.read_point_cloud(source_file)
    target = o3d.io.read_point_cloud(target_file)

    # Show models side by side
    #draw_registrations(source, target)

    source_d, source_features = preprocess_point_cloud(source, voxel_size)
    target_d, target_features = preprocess_point_cloud(target, voxel_size)
    print(source_file, "->", target_file)
    print(source_features)

    ransac_result_ptp = reg.registration_ransac_based_on_feature_matching(
        source_d, target_d, 
        source_features, target_features, 
        max_correspondence_distance = voxel_size * 1.5,
        estimation_method=point_to_point, mutual_filter=True,
        checkers=[
        reg.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        reg.CorrespondenceCheckerBasedOnDistance(voxel_size * 1.5),
        ],
        criteria=reg.RANSACConvergenceCriteria(100000, 0.9999))

    #Evaluate registration
    print("Ransac alignment")
    evaluation = reg.evaluate_registration(source, target, threshold, ransac_result_ptp.transformation)
    print("Fitness:", evaluation.fitness)
    print("RMSE:", evaluation.inlier_rmse)

    # Added: the same evaluation at the distance RANSAC itself used (voxel_size * 1.5)
    evaluation_ransac = reg.evaluate_registration(source, target, voxel_size * 1.5, ransac_result_ptp.transformation)
    print("Fitness at RANSAC threshold (%.3f):" % (voxel_size * 1.5), evaluation_ransac.fitness)
    print()

    results[source_file] = (source, target, ransac_result_ptp)
    #draw_registrations(source, target, ransac_result_ptp.transformation, True)

global_registration/r1.pcd -> global_registration/r2.pcd
Feature class with dimension = 33 and num = 4760
Access its data via data member.
Ransac alignment
Fitness: 0.5600170996052003
RMSE: 0.010096892394147416
Fitness at RANSAC threshold (0.075): 0.658877964141122

global_registration/car1.ply -> global_registration/car2.ply
Feature class with dimension = 33 and num = 3490
Access its data via data member.
Ransac alignment
Fitness: 0.9619776438171558
RMSE: 0.008429174541009022
Fitness at RANSAC threshold (0.075): 0.999651060655284



I notice in the print above that the fitness varies ALOT. With current parameters it goes between 0.4 and 0.6 when using the office room. It shows ransac really is random.

To make it easier to compare runs we seed the random generator in the first cell (`o3d.utility.random.seed(42)`). This reduces the variation, but open3d runs RANSAC on several threads, so the numbers can still change a little between runs.

Fitness counts the share of points that have a match closer than the threshold, so it depends on the threshold that is chosen. That is why the second fitness line, evaluated at the same distance RANSAC used (0.075), is higher than the one at 0.02.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [5]:
#Code is copy paste from notebook on monday

point_to_point =  reg.TransformationEstimationPointToPoint()

for source_file, (source, target, ransac_result_ptp) in results.items():
    target.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=5,
            max_nn=30
        )
    )
    source.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=5,
            max_nn=30
        )
    )


    # ransac_result_ptp.transformation is the result from task 1, use that as initial transform
    icp_result = reg.registration_icp(
        source, target, threshold,  ransac_result_ptp.transformation,
        point_to_point)
        
    #Evaluate registration
    print(source_file)
    print("Final alignment using ICP")
    evaluation = reg.evaluate_registration(source, target, threshold, icp_result.transformation)
    print("Fitness:", evaluation.fitness)
    print("RMSE:", evaluation.inlier_rmse)
    print()
    #draw_registrations(source, target, icp_result.transformation, True)

global_registration/r1.pcd
Final alignment using ICP
Fitness: 0.621143158900596
RMSE: 0.006564491232725272

global_registration/car1.ply
Final alignment using ICP
Fitness: 0.9863251874044929
RMSE: 0.005406814686113036



#### Additional ICP experiments
Below we also compare point-to-point with point-to-plane ICP, and the threshold of 0.02 with a looser 0.05 (RANSAC is only accurate to roughly one voxel, 0.05, so a looser threshold lets ICP find more matching pairs at the start).

Point-to-plane needs a normal for each point that describes the local surface. The clouds are in metres and the objects are 2-3 m across, so the normals are estimated from a small neighbourhood (radius 0.1, i.e. 2x voxel_size).

Every result is printed with the fitness and RMSE at the threshold that ICP used, next to the RANSAC result evaluated at the same threshold.

In [6]:
icp_results = {}
for source_file, (source, target, ransac_result_ptp) in results.items():
    for pcd in (source, target):
        pcd.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
        )

    print(source_file)
    for icp_threshold in [0.02, 0.05]:
        before = reg.evaluate_registration(source, target, icp_threshold, ransac_result_ptp.transformation)
        print(f"  threshold {icp_threshold}:")
        print(f"    RANSAC only     fitness={before.fitness:.3f}  RMSE={before.inlier_rmse:.4f}")
        for name, method in [("point-to-point", reg.TransformationEstimationPointToPoint()),
                             ("point-to-plane", reg.TransformationEstimationPointToPlane())]:
            icp = reg.registration_icp(
                source, target, icp_threshold, ransac_result_ptp.transformation, method)
            print(f"    ICP {name}  fitness={icp.fitness:.3f}  RMSE={icp.inlier_rmse:.4f}")
            icp_results[(source_file, name, icp_threshold)] = icp
    print()

global_registration/r1.pcd
  threshold 0.02:
    RANSAC only     fitness=0.560  RMSE=0.0101
    ICP point-to-point  fitness=0.621  RMSE=0.0066
    ICP point-to-plane  fitness=0.621  RMSE=0.0066
  threshold 0.05:
    RANSAC only     fitness=0.646  RMSE=0.0145
    ICP point-to-point  fitness=0.647  RMSE=0.0093
    ICP point-to-plane  fitness=0.646  RMSE=0.0093

global_registration/car1.ply
  threshold 0.02:
    RANSAC only     fitness=0.962  RMSE=0.0084
    ICP point-to-point  fitness=0.986  RMSE=0.0054
    ICP point-to-plane  fitness=0.988  RMSE=0.0054
  threshold 0.05:
    RANSAC only     fitness=0.999  RMSE=0.0097
    ICP point-to-point  fitness=0.999  RMSE=0.0062
    ICP point-to-plane  fitness=0.999  RMSE=0.0063



Results of the additional experiments (the car's RANSAC numbers can change a little between runs, see above):

- ICP improves both pairs. For the office room at threshold 0.02, fitness goes from 0.31 after RANSAC to 0.62 after ICP and the RMSE drops from 0.013 to 0.007. The car was already almost perfectly aligned by RANSAC (fitness 0.98) and ICP brings it to 0.99.
- Point-to-point and point-to-plane end up at practically the same result here, because RANSAC already gives a good starting point.
- The threshold of 0.02 works as well as 0.05. The fitness is higher at 0.05 only because more points count as a match, the alignment itself is about the same.
- The office room never reaches a fitness close to 1 because the two scans only partly overlap, so a part of each scan has nothing to match with.

#### Task 2b 
Function for calculating cloud points of the car. It Takes 2 arguments first one is step(default value is 1) which will skip this amount of frames. We skip images to reduce computation time in trade for accuracy. The second argument is number of images (default value is 1722) in dataset. It is used to indicate number of image we want to traverse and include in calculations. What is worth mentioning is that voxel_size is used in different functions and it is tuned accordingly to trial and error steps. In general code calculate match between current cloud points and previous. Firstly it uses local ICP matching algorithm. If the result is in the desired threshold it composes transformation in global position and modifies current_world by this which is deep copy of the current_cloud. Later it is added to the accumulated variable which is our final result. If the threshold was not reached we calculated global matching by RANSAC and later we do ICP again to get the best result, but if the result is not better than first ICP calculation we just use the first result as our transformation. The problem here is that each frame only compare it selve to the precedecessor so at the end the global error is rising. This is why on the final result we can see a "ghost" car which comes from the main part and other variances. To fix that we would need to compare frames which are further to each other. 

In [ ]:
def retrieve_point_cloud(step=1, number_of_images=1722):
    voxel_size = 0.05
    camera = o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault
    )

    #function to read point cloud from rgb image and depth
    def load_point_cloud(idx):
        img = o3d.io.read_image(f'car_challange/rgb/{idx:07d}.jpg')
        depth = o3d.io.read_image(f'car_challange/depth/{idx:07d}.png')
        rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
            img, depth, convert_rgb_to_intensity=False
        )
        pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, camera)
        #rotating point cloud so it is not rotated
        pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
        pcd = pcd.voxel_down_sample(voxel_size)
        pcd.estimate_normals(
            o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30)
        )
        return pcd

    point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()
    point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint()

    #loading first point cloud from the dataset so it can be later compared with next point cloud
    prev_cloud = load_point_cloud(1)
    global_pose = np.identity(4)
    #variable which will store final result, deepcopy is used so every operation on accumulated doesn't change prev_cloud
    accumulated = copy.deepcopy(prev_cloud)
    icp_fitness_threshold = 0.8   #if the first local matching has bad result this threshold starts global alignment first to improve result

    for idx in range(2, number_of_images + 1, step):
        #loading currect point cloud so it can be comared with previous point cloud
        current_cloud = load_point_cloud(idx)

        icp_result = o3d.pipelines.registration.registration_icp(
            current_cloud, prev_cloud, voxel_size * 0.8, np.identity(4), point_to_plane
        )

        if icp_result.fitness < icp_fitness_threshold:
            print(f"Frame {idx} requires global matching (fitness={icp_result.fitness:.3f})")

            feature_current = o3d.pipelines.registration.compute_fpfh_feature(
                current_cloud, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 10, max_nn=100)
            )
            feature_prev = o3d.pipelines.registration.compute_fpfh_feature(
                prev_cloud, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 10, max_nn=100)
            )

            ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
                current_cloud, prev_cloud, feature_current, feature_prev,
                True, voxel_size * 2, point_to_point, 3, [],
                o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999)
            )
            icp_new_result = o3d.pipelines.registration.registration_icp(
                current_cloud, prev_cloud, voxel_size * 0.8, ransac_result.transformation, point_to_plane
            )

            if icp_result.fitness <= icp_new_result.fitness:
                icp_result = icp_new_result
            else:
                print(f"Frame {idx} reusing previous icp result")


        # Composing transformation in order not to do transformation from the beginning but the latest point
        global_pose = global_pose @ icp_result.transformation

        current_world = copy.deepcopy(current_cloud)
        current_world.transform(global_pose)

        #accumulating new point cloud in our final result variable
        accumulated += current_world
        #down sampling new result for better performence
        accumulated = accumulated.voxel_down_sample(voxel_size)

        #Chaning prev to current for further calculations
        prev_cloud = current_cloud

    return accumulated

In [8]:
accumulated_cloud = retrieve_point_cloud(step=5, number_of_images=1722)

#Visualizing final result
o3d.visualization.draw_geometries([accumulated_cloud])

frame 347 requires global matching (fitness=0.784)
Frame 347 reusing previous icp result
frame 972 requires global matching (fitness=0.799)
frame 1332 requires global matching (fitness=0.790)
frame 1337 requires global matching (fitness=0.746)
frame 1372 requires global matching (fitness=0.800)
frame 1377 requires global matching (fitness=0.756)
frame 1382 requires global matching (fitness=0.599)
frame 1387 requires global matching (fitness=0.722)
frame 1392 requires global matching (fitness=0.766)
frame 1432 requires global matching (fitness=0.776)
frame 1437 requires global matching (fitness=0.724)
frame 1442 requires global matching (fitness=0.705)
frame 1462 requires global matching (fitness=0.760)
frame 1482 requires global matching (fitness=0.618)
frame 1492 requires global matching (fitness=0.540)
frame 1497 requires global matching (fitness=0.676)
frame 1507 requires global matching (fitness=0.680)
[Open3D WARNING] Too few correspondences (20) after mutual filter, fall back to 

## Screenshots
Most of the visualisation in this project happens in a separate 3D window, so the cell below saves screenshots of the results to the `screenshots/` folder for the hand-in. For each pair it saves the input (before alignment), the alignment after RANSAC and the alignment after ICP (point-to-plane, threshold 0.05). The source is painted orange and the target blue, like `draw_registrations(..., recolor=True)`.

In [7]:
import os

os.makedirs("screenshots", exist_ok=True)

# Same as draw_registrations, but renders in a hidden window and saves a PNG
# source_view=True moves the target onto the source (inverse transform) instead of the
# other way round. Same alignment, but the camera then uses the source's viewpoint,
# which shows the car from a readable angle (from the target's side it is edge-on).
def save_registration(source, target, transformation, filename, source_view=False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    if transformation is not None and source_view:
        target_temp.transform(np.linalg.inv(transformation))
    elif transformation is not None:
        source_temp.transform(transformation)

    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=False, width=1280, height=960)
    vis.add_geometry(source_temp)
    vis.add_geometry(target_temp)
    vis.get_render_option().point_size = 2
    vis.poll_events()
    vis.update_renderer()
    vis.capture_screen_image(filename, do_render=True)
    vis.destroy_window()
    print("saved", filename)

for source_file, (source, target, ransac_result_ptp) in results.items():
    name = "car" if "car" in source_file else "office"
    icp = icp_results[(source_file, "point-to-plane", 0.05)]
    source_view = (name == "car")
    save_registration(source, target, None, f"screenshots/{name}_1_before.png", source_view)
    save_registration(source, target, ransac_result_ptp.transformation, f"screenshots/{name}_2_ransac.png", source_view)
    save_registration(source, target, icp.transformation, f"screenshots/{name}_3_icp.png", source_view)

saved screenshots/office_1_before.png


saved screenshots/office_2_ransac.png


saved screenshots/office_3_icp.png
saved screenshots/car_1_before.png


saved screenshots/car_2_ransac.png
saved screenshots/car_3_icp.png
